In [ ]:
import datetime as dt
import os
import sys

sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))

import concurrent.futures
import itertools
from multiprocessing import cpu_count

import numpy as np
import statsmodels.api as sm
from dotenv import load_dotenv
from mc_postgres_db import models as mc
from sqlalchemy import create_engine, select
from sqlalchemy.orm import Session, aliased
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from tqdm.notebook import tqdm
import polars as pl
import scipy.optimize as so
import pandas as pd
import matplotlib.pyplot as plt

## Monte-carlo Pairs Trading

For this model, I want the model to take into account N crypto currencies and re-balance a portfolio for some frequency based on a pairs trading strategy. I want the strategy to dynamically search over all the currencies and group them into pairs if they are correlated by a certain amount, disregarding pairs that have the lowest correlation.

Based on the above, we can simulate Ornstein-Uhlenbeck (OU) processes for each pair simulatenously similar to what is described in [1]. For a pair of assets $S_1$ and $S_2$, we have the following equation that defines their relationship to one another:

$$
X_t = \alpha S^1_t - \beta S^2_t \tag{1}
$$

The constants $\alpha$ and $\beta$ will vary depending on the specific pair we are looking at.

$$
dX_t = \mu (\theta - X_t)dt + \sigma d W_t \tag{2}
$$

Where $\mu$ is the mean-reverting constant, $\theta$ is the long-term mean, and $\sigma$ is the standard deviation and are assumed to be constant. In reality, these valuesw will likely change over time. Because of this in our actual model we will re-calculate these parameters on a rolling window. The final variable $W_t$ is a standard Brownian motion under $\mathbb{P}$. As per, the paper [here](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2222196), we have the general solution to this differential equation.

$$
\begin{align}
f^{OU}(x_i^{\alpha,\beta}|x_{i-1}^{\alpha,\beta};\theta,\mu,\sigma) = \frac{1}{\sqrt{2 \pi \tilde{\sigma}^2}} \exp{\left(-\frac{(x_i - x_{i-1}e^{-\mu \Delta t} - \theta (1 - e^{-\mu \Delta t}))^2}{2\tilde{\sigma}^2}\right)} \tag{3}
\end{align}
$$

With the constant:

$$
\tilde{\sigma}^2 = \sigma^2 \frac{1 - e^{-2\mu \Delta t}}{2 \mu} \tag {4}
$$

Before we will leverage this equation, I want to fit our model to actual data by finding $\alpha$, $\beta$, $\theta$, $\mu$, and $\sigma$ using the maximized average log-likelihood. We will start by finding all cointegrated paris over the past 7 days.

In [ ]:
# Initialize the database connection.
load_dotenv()
POSTGRES_URL = os.getenv("POSTGRES_URL")
engine = create_engine(POSTGRES_URL)

# Define the start and end dates.
lookback_period = 30
end_date = dt.datetime(2025, 1, 1)
query_start_date = end_date - dt.timedelta(days=lookback_period + 1)
start_date = query_start_date + dt.timedelta(days=1)

# Get the Kraken provider.
with Session(engine) as session:
    stmt = select(mc.Provider).where(mc.Provider.name == "Kraken")
    kraken = session.execute(stmt).scalar_one()

# Get the USD asset.
with Session(engine) as session:
    stmt = select(mc.Asset).where(mc.Asset.name == "USD")
    usd = session.execute(stmt).scalar_one()

# Get all asset pairs from Kraken.
from_asset = aliased(mc.Asset)
to_asset = aliased(mc.Asset)
df = pl.read_database(
    select(
        mc.ProviderAssetMarket.timestamp,
        mc.ProviderAssetMarket.from_asset_id,
        mc.ProviderAssetMarket.to_asset_id,
        from_asset.name.label("from_asset_name"),
        to_asset.name.label("to_asset_name"),
        mc.ProviderAssetMarket.close,
    )
    .join(to_asset, mc.ProviderAssetMarket.to_asset_id == to_asset.id)
    .join(from_asset, mc.ProviderAssetMarket.from_asset_id == from_asset.id)
    .where(
        mc.ProviderAssetMarket.provider_id == kraken.id,
        mc.ProviderAssetMarket.from_asset_id == usd.id,
        mc.ProviderAssetMarket.timestamp >= query_start_date,
        mc.ProviderAssetMarket.timestamp <= end_date,
        to_asset.underlying_asset_id == None,
        to_asset.asset_type_id != usd.id,
    )
    .order_by(mc.ProviderAssetMarket.timestamp),
    engine,
)
display(df)

In [ ]:
# Generate a dataframe of all the timestamps and the assets.
frame = pl.DataFrame(
    {
        "timestamp": pl.datetime_range(
            start=query_start_date, end=end_date, interval="1m", eager=True
        )
    }
).join(
    df.filter(pl.col("timestamp") >= start_date)
    .filter(pl.col("timestamp") <= end_date)[
        ["from_asset_id", "from_asset_name", "to_asset_id", "to_asset_name"]
    ]
    .unique(),
    how="cross",
)

# Join the dataframe with the asset pairs.
df = frame.join(df, on=["timestamp", "to_asset_id", "from_asset_id"], how="left")
df = (
    df.group_by(["to_asset_id", "from_asset_id", "to_asset_name", "from_asset_name"])
    .agg(
        pl.col("timestamp").sort_by("timestamp"),
        pl.col("close").sort_by("timestamp").forward_fill().alias("close"),
    )
    .explode(["timestamp", "close"])
)

# Filter the dataframe to the desired timeframe.
df = df.filter(pl.col("timestamp") >= start_date).filter(
    pl.col("timestamp") <= end_date
)[
    [
        "timestamp",
        "to_asset_id",
        "to_asset_name",
        "from_asset_id",
        "from_asset_name",
        "close",
    ]
]
display(df)

In [ ]:
df.filter(pl.any_horizontal(pl.col("close").is_null(), pl.col("close").is_null()))

In [ ]:
df = df.filter(pl.col("close").is_not_null())

Next we will take this data and determine the cross-product of every asset, only producing unique combinations and generate a dataframe with all of these pairs and their close prices.

In [ ]:
assets = df["to_asset_id"].unique().to_list()
print(f"Number of assets: {len(assets):,}")
asset_combinations = list(itertools.combinations(assets, 2))
print(f"Number of combinations: {len(asset_combinations):,}")
asset_combinations_df = pl.DataFrame(
    [
        {"to_asset_id_1": pair[0], "to_asset_id_2": pair[1]}
        for pair in asset_combinations
    ]
)
cdf = (
    df[["timestamp"]]
    .unique()
    .sort("timestamp")
    .join(asset_combinations_df, how="cross")
)
cdf = cdf.join(
    df.rename(
        {
            "from_asset_id": "from_asset_id_1",
            "to_asset_id": "to_asset_id_1",
            "from_asset_name": "from_asset_name_1",
            "to_asset_name": "to_asset_name_1",
            "close": "close_1",
        }
    ),
    on=["timestamp", "to_asset_id_1"],
    how="left",
)
cdf = cdf.join(
    df.rename(
        {
            "from_asset_id": "from_asset_id_2",
            "to_asset_id": "to_asset_id_2",
            "from_asset_name": "from_asset_name_2",
            "to_asset_name": "to_asset_name_2",
            "close": "close_2",
        }
    ),
    on=["timestamp", "to_asset_id_2"],
    how="left",
)
cdf = cdf[
    [
        "timestamp",
        "to_asset_id_1",
        "to_asset_name_1",
        "close_1",
        "to_asset_id_2",
        "to_asset_name_2",
        "close_2",
    ]
]
cdf = cdf.select(
    [
        "timestamp",
        "to_asset_id_1",
        "to_asset_name_1",
        "close_1",
        "to_asset_id_2",
        "to_asset_name_2",
        "close_2",
    ]
)
cdf = cdf.filter(pl.col("timestamp") >= start_date).filter(
    pl.col("timestamp") <= end_date
)
print(f"Number of rows: {len(cdf):,}")

In [ ]:
cdf.write_parquet("cdf.parquet")

In [ ]:
cdf = pl.read_parquet("cdf.parquet")

Using the above, we will compute the cointegration for each pair and filter ones that have a p-value lower than some threshold.

In [ ]:
cointegration_df = (
    cdf.group_by(
        [
            pl.col("to_asset_id_1"),
            pl.col("to_asset_name_1"),
            pl.col("to_asset_id_2"),
            pl.col("to_asset_name_2"),
        ]
    )
    .agg(
        pl.col("close_1").sort_by("timestamp"),
        pl.col("close_2").sort_by("timestamp"),
    )
    .sample(10)
)

# Prepare data for parallel processing
close_1_arrays = cointegration_df["close_1"].to_numpy()
close_2_arrays = cointegration_df["close_2"].to_numpy()
data_pairs = list(zip(close_1_arrays, close_2_arrays))

# Compute the cointegration statistics for each pair.
max_workers = int(cpu_count() * 0.25)
print(f"Using {max_workers} workers.")


def coint_stats(pair: tuple[np.ndarray, np.ndarray]) -> dict:
    """
    Compute the cointegration statistics for a pair of assets.
    """

    # Get the two assets.
    X, y = pair

    # Compute the OLS model.
    X = sm.add_constant(X)
    model = sm.OLS(y, X)
    results = model.fit()
    alpha = results.params[0]
    beta = results.params[1]

    # Compute the adfuller test.
    adf, pvalue, usedlag, nobs, critical_values, icbest = adfuller(results.resid)

    # Get the cointegration statistics.
    return {
        "alpha": alpha,
        "beta": beta,
        "adf": adf,
        "pvalue": pvalue,
        "usedlag": usedlag,
        "nobs": nobs,
        "critical_value_1pct": critical_values["1%"],
        "critical_value_5pct": critical_values["5%"],
        "critical_value_10pct": critical_values["10%"],
        "icbest": icbest,
        "residuals": results.resid,
    }


# Compute the cointegration statistics for each pair.
with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
    results = list(
        tqdm(
            executor.map(coint_stats, data_pairs),
            total=len(data_pairs),
        )
    )

# Create a dataframe from the results.
cointegration_df = cointegration_df.with_columns(
    [
        pl.Series(name="alpha", values=[r["alpha"] for r in results]),
        pl.Series(name="beta", values=[r["beta"] for r in results]),
        pl.Series(name="adf", values=[r["adf"] for r in results]),
        pl.Series(name="p_value", values=[r["pvalue"] for r in results]),
        pl.Series(name="usedlag", values=[r["usedlag"] for r in results]),
        pl.Series(name="nobs", values=[r["nobs"] for r in results]),
        pl.Series(
            name="critical_value_1pct",
            values=[r["critical_value_1pct"] for r in results],
        ),
        pl.Series(
            name="critical_value_5pct",
            values=[r["critical_value_5pct"] for r in results],
        ),
        pl.Series(
            name="critical_value_10pct",
            values=[r["critical_value_10pct"] for r in results],
        ),
        pl.Series(name="icbest", values=[r["icbest"] for r in results]),
        pl.Series(name="residuals", values=[r["residuals"] for r in results]),
    ]
).drop(["close_1", "close_2"])

# Filter the cointegrated pairs.
threshold = 0.01
cointegrated_pairs_df = cointegration_df.filter(pl.col("p_value") < threshold).sort(
    "p_value"
)
print(f"Number of cointegrated pairs: {len(cointegrated_pairs_df)}")

In [ ]:
cointegration_df.write_parquet("cointegration_df.parquet")

In [ ]:
cointegration_df = pl.read_parquet("cointegration_df.parquet")

# Filter the cointegrated pairs.
threshold = 0.01
cointegrated_pairs_df = cointegration_df.filter(pl.col("p_value") < threshold).sort(
    "p_value"
)
print(f"Number of cointegrated pairs: {len(cointegrated_pairs_df)}")

From these cointegrated pairs we then fit the OU equation in [3] for $\mu$, $\theta$, and $\sigma$ using the average log-likelihood defined by:

$$
\begin{align}
\ell(\theta,\mu,\sigma|x_0^{\alpha,\beta},x_1^{\alpha,\beta},\ldots,x_n^{\alpha,\beta}) &= \frac{1}{n} \sum_{i=1}^n \ln{f^{OU}(x_i^{\alpha,\beta}|x_{i-1}^{\alpha,\beta};\theta,\mu,\sigma)} \tag{5} \\
&= - \frac{1}{2} \ln{2\pi} - \ln{\tilde{\sigma}} - \frac{1}{2 n \tilde{\sigma}^2} \sum_{i=1}^{n} \left[x_i^{\alpha,\beta} - x_{i-1}^{\alpha,\beta} e^{-\mu \Delta t}- \sigma(1 - e^{-\mu \Delta t})  \right]^2 \tag{6}
\end{align}
$$

In [ ]:
def fit_ou_parameters(X: np.ndarray, dt: float):
    model_arima = ARIMA(X, order=(1, 0, 0))
    alpha, phi, sigma_epsilon_squared = model_arima.fit(return_params=True)
    dt = 1 / len(X)
    theta = (1 - phi) / dt  # mean-reverting speed
    sigma = np.sqrt(sigma_epsilon_squared) / np.sqrt(dt)  # volatility
    mu = np.mean(X)  # long-term mean from data
    return theta, mu, sigma

In [ ]:
sample = cointegrated_pairs_df.to_pandas().iloc[0]
alpha = sample["alpha"]
beta = sample["beta"]
X = sample["residuals"]
delta_t = 1 / len(X)
theta, mu, sigma = fit_ou_parameters(X, delta_t)

print(f"theta: {theta}")
print(f"sigma: {sigma}")
print(f"mu: {mu}")

In [ ]:
N_simulated = 100
N = len(X)
X_simulated = np.zeros((N_simulated, N))
X_simulated[:, 0] = 0  # initial value

for i in range(1, N):
    X_simulated[:, i] = (
        X_simulated[:, i - 1] * np.exp(-theta * delta_t)
        + mu * (1 - np.exp(-theta * delta_t))
        + sigma
        * np.sqrt((1 - np.exp(-2 * theta * delta_t)) / (2 * theta))
        * np.random.normal(0, 1, N_simulated)
    )

thetas = np.zeros(N_simulated)
mus = np.zeros(N_simulated)
sigmas = np.zeros(N_simulated)
for i in tqdm(range(N_simulated)):
    thetas[i], mus[i], sigmas[i] = fit_ou_parameters(X_simulated[i, :], 1)

In [ ]:
for i in range(N_simulated):
    plt.plot(X_simulated[i], alpha=0.15)
plt.plot(X)
plt.show()

In [ ]:
simulation_results = pd.DataFrame(
    {
        "name": ["Empirical", "Simulated"],
        "theta": [theta, thetas.mean()],
        "mu": [mu, mus.mean()],
        "sigma": [sigma, sigmas.mean()],
    }
).set_index("name")
print(simulation_results.to_string())

In [ ]:
class OrnsteinUhlenbeck:
    X: np.ndarray
    dt: float

    def __init__(self, X: np.ndarray, dt: float):
        self.X = X
        self.dt = dt

    @staticmethod
    def __log_likelihood(
        params: tuple[float, float, float], X: np.ndarray, dt: float
    ) -> float:
        theta, mu, sigma = params
        n = len(X)
        X_lag = X[:-1]
        X_next = X[1:]
        tilde_sigma = sigma * np.sqrt((1 - np.exp(-2 * mu * dt)) / (2 * mu))
        log_likelihood = (
            -0.5 * np.log(2 * np.pi)
            - np.log(tilde_sigma)
            - 1
            / (2 * n * tilde_sigma**2)
            * np.sum(
                (X_next - X_lag * np.exp(-mu * dt) - theta * (1 - np.exp(-mu * dt)))
                ** 2
            )
        )
        return -log_likelihood

    def fit(self):
        """
        Estimates Ornstein-Uhlenbeck coefficients (θ, µ, σ) of the given array
        using the Maximum Likelihood Estimation method

        input: X - array-like data to be fit as an OU process
        returns: θ, µ, σ, Total Log Likelihood
        """
        small_bound = 1e-9
        bounds = (
            (None, None),
            (small_bound, None),
            (small_bound, None),
        )  # theta > 0, mu ∈ ℝ, sigma > 0
        mu_init = small_bound
        sigma_init = np.std(self.X)
        theta_init = np.mean(self.X)
        result = so.minimize(
            OrnsteinUhlenbeck.__log_likelihood,
            (theta_init, mu_init, sigma_init),
            args=(self.X, self.dt),
            bounds=bounds,
            tol=1e-10,
        )
        theta, mu, sigma = result.x
        max_log_likelihood = -result.fun  # undo negation from __compute_log_likelihood
        return theta, mu, sigma, max_log_likelihood

In [ ]:
sample = cointegrated_pairs_df.to_pandas().iloc[1]
alpha = sample["alpha"]
beta = sample["beta"]
X = sample["residuals"]
dt = 1
result = OrnsteinUhlenbeck(X, dt).fit()
theta, mu, sigma, max_log_likelihood = result

print(f"theta: {theta}")
print(f"mu: {mu}")
print(f"sigma: {sigma}")
print(f"max_log_likelihood: {max_log_likelihood}")

In [ ]:
N_simulated = 1000
X_simulated = np.zeros((N_simulated, len(X)))
X_simulated[:, 0] = 0  # initial value

for i in range(1, len(X)):
    X_simulated[:, i] = (
        X_simulated[:, i - 1] * np.exp(-theta * dt)
        + mu * (1 - np.exp(-theta * dt))
        + sigma
        * np.sqrt((1 - np.exp(-2 * theta * dt)) / (2 * theta))
        * np.random.normal(0, 1, N_simulated)
    )


for i in range(N_simulated):
    plt.plot(X_simulated[i], alpha=0.25)
plt.plot(X)
plt.show()

In [ ]:
# Compute the OU parameters for each pair.
ou_parameters_df = (
    cdf.group_by(
        [
            pl.col("to_asset_id_1"),
            pl.col("to_asset_name_1"),
            pl.col("to_asset_id_2"),
            pl.col("to_asset_name_2"),
        ]
    )
    .agg(
        pl.col("close_1").sort_by("timestamp"),
        pl.col("close_2").sort_by("timestamp"),
    )
    .sample(10)
)

# Organize the data for parallel processing.
close_1_arrays = ou_parameters_df["close_1"].to_numpy()
close_2_arrays = ou_parameters_df["close_2"].to_numpy()
data_pairs = list(zip(close_1_arrays, close_2_arrays))


def compute_ou_parameters(data_pair):
    S_1 = data_pair[0]
    S_2 = data_pair[1]
    B = np.linspace(-1, 1, 100)
    betas = np.zeros(len(B))
    max_log_likelihoods = np.zeros(len(B))
    argmax_beta = -np.inf
    argmax_mu = -np.inf
    argmax_sigma = -np.inf
    argmax_theta = -np.inf
    for j, B_i in enumerate(B):
        alpha = 1 / S_1[0]
        betas[j] = B_i / S_2[0]
        X = alpha * S_1 - betas[j] * S_2
        ou = OrnsteinUhlenbeck(X)
        theta, mu, sigma, max_log_likelihood = ou.fit()
        max_log_likelihoods[j] = max_log_likelihood
        if max_log_likelihood > argmax_beta:
            argmax_beta = betas[j]
            argmax_mu = mu
            argmax_sigma = sigma
            argmax_theta = theta
    return (
        argmax_beta,
        argmax_mu,
        argmax_sigma,
        argmax_theta,
        betas,
        max_log_likelihoods,
    )


# Compute the OU parameters for each pair.
max_workers = int(cpu_count() * 0.75)
print(f"Using {max_workers} workers.")
with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
    results = list(
        tqdm(
            executor.map(compute_ou_parameters, data_pairs),
            total=len(data_pairs),
        )
    )

# Create a dataframe from the results.
ou_parameters_df = ou_parameters_df.with_columns(
    [
        pl.Series(name="argmax_beta", values=[r[0] for r in results]),
        pl.Series(name="argmax_mu", values=[r[1] for r in results]),
        pl.Series(name="argmax_sigma", values=[r[2] for r in results]),
        pl.Series(name="argmax_theta", values=[r[3] for r in results]),
        pl.Series(name="betas", values=[r[4] for r in results]),
        pl.Series(name="max_log_likelihoods", values=[r[5] for r in results]),
    ]
)

In [ ]:
# Sample the first row.
sample = ou_parameters_df.to_pandas().iloc[1]
beta = sample["argmax_beta"]
mu = sample["argmax_mu"]
sigma = sample["argmax_sigma"]
theta = sample["argmax_theta"]
betas = sample["betas"]
alpha = 1 / sample["close_1"][0]
max_log_likelihoods = sample["max_log_likelihoods"]
close_1 = sample["close_1"]
close_2 = sample["close_2"]
delta_t = 1 / len(close_1)

# Calculate the actual X.
X = alpha * close_1 - beta * close_2

# Filt to the AR(1) process.
model_arima = ARIMA(X, order=(1, 0, 0))
results_arima = model_arima.fit()

phi = results_arima.params[0]
sigma = np.sqrt(results_arima.sigma2)

results_arima.summary()

In [ ]:
# Sample the first row.
sample = ou_parameters_df.to_pandas().iloc[0]
beta = sample["argmax_beta"]
mu = sample["argmax_mu"]
sigma = sample["argmax_sigma"]
theta = sample["argmax_theta"]
betas = sample["betas"]
alpha = 1 / sample["close_1"][0]
max_log_likelihoods = sample["max_log_likelihoods"]
close_1 = sample["close_1"]
close_2 = sample["close_2"]
delta_t = 1 / len(close_1)

print("\nActual OU Process:")
print(f"theta: {theta}")
print(f"mu: {mu}")
print(f"sigma: {sigma}")
print(f"max_log_likelihood: {max_log_likelihoods[np.argmax(max_log_likelihoods)]}")

# Calculate the actual X.
X = alpha * close_1 - beta * close_2

# Filt to the AR(1) process.
model_arima = ARIMA(X, order=(1, 0, 0))
results_arima = model_arima.fit()

# Generate the OU process based on the parameters.
X_simulated = np.zeros(len(close_1))
X_simulated[0] = 0
for i in range(1, len(close_1)):
    X_simulated[i] = mu * (theta - X_simulated[i - 1]) + sigma * np.random.normal(
        0, np.sqrt(delta_t)
    )

# Plot the simulated OU process.
plt.plot(X_simulated)
plt.show()

# Check the fit of the simulated OU process.
ou_simulated = OrnsteinUhlenbeck(X_simulated)
theta_simulated, mu_simulated, sigma_simulated, max_log_likelihood_simulated = (
    ou_simulated.fit()
)
print("\nSimulated OU Process:")
print(f"theta (simulated): {theta_simulated}")
print(f"mu (simulated): {mu_simulated}")
print(f"sigma (simulated): {sigma_simulated}")
print(f"max_log_likelihood (simulated): {max_log_likelihood_simulated}")